# Active learning with STACNotator

This notebook walks through one full active-learning round against a STACNotator campaign:

1. Log in via the browser and open a campaign
2. Fetch the labeled samples and split them into train/test
3. Train a deliberately simple model (lat/lon as the only features)
4. Predict over a pixel grid covering the campaign extent and write the result as a COG
5. Register the prediction as a map layer with a class legend, so annotators see where the model is confused
6. After annotators add labels, grow the training set with `update_samples` and retrain

The lat/lon model is a stand-in: it only demonstrates the data plumbing. In a real setup you
would featurize with raster bands or embeddings; loading raster data through the SDK is on the
roadmap (see the README TODOs).

In [2]:
# The SDK is not on PyPI yet; install it from this repo checkout
# (this notebook lives in sdk/examples/, the package root sdk/ is one dir up).
%pip install -q .. scikit-learn rasterio matplotlib

ERROR: Could not find a version that satisfies the requirement stacnotator-sdk (from versions: none)
ERROR: No matching distribution found for stacnotator-sdk
Note: you may need to restart the kernel to use updated packages.


## 1. Login and pick a campaign

`login` opens your browser once and caches the credential locally. Against a local dev
backend (`AUTH_PROVIDER=local`) it just works without a browser.

In [ ]:
import stacnotator as snt

URL = "http://localhost:5173"  # your STACNotator app URL

snt.login(URL)
snt.campaigns()

In [ ]:
CAMPAIGN_ID = None  # pick one from the table above

campaign = snt.campaign(CAMPAIGN_ID)
print(campaign)
print("labels:", campaign.labels)
print("extent:", campaign.extent)

## 2. Fetch samples and split

One row per labeled annotation. The test set is held out once and stays fixed for the whole
campaign, so scores are comparable across rounds.

In [ ]:
from sklearn.model_selection import train_test_split

samples = campaign.get_samples()
print(f"{len(samples)} labeled samples")
samples.head()

In [ ]:
train, test = train_test_split(samples, test_size=0.2, random_state=42, stratify=samples["label_id"])
len(train), len(test)

## 3. Train a very simple model

Point samples only, and the features are just lat/lon.

In [ ]:
from sklearn.ensemble import RandomForestClassifier


def featurize(df):
    points = df.dropna(subset=["lat", "lon"])
    return points[["lon", "lat"]].to_numpy(), points["label_id"].to_numpy(dtype="int64")


model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(*featurize(train))
print(f"held-out accuracy: {model.score(*featurize(test)):.2f}")

## 4. Predict over the campaign extent

First a quick visual sanity check on random pixels, then a regular grid that becomes the
prediction raster.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

west, south, east, north = campaign.extent

rng = np.random.default_rng(42)
random_pixels = np.column_stack(
    [rng.uniform(west, east, 2000), rng.uniform(south, north, 2000)]
)
predicted = model.predict(random_pixels)

fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(random_pixels[:, 0], random_pixels[:, 1], c=predicted, s=4, cmap="tab10")
ax.legend(scatter.legend_elements()[0], [campaign.labels.get(i, i) for i in np.unique(predicted)])
ax.set_title("Model predictions at random pixels in the extent")
plt.show()

In [ ]:
WIDTH = HEIGHT = 512

lons = np.linspace(west, east, WIDTH)
lats = np.linspace(north, south, HEIGHT)  # north up: first row is the top
grid = np.column_stack([np.tile(lons, HEIGHT), np.repeat(lats, WIDTH)])

class_raster = model.predict(grid).reshape(HEIGHT, WIDTH).astype("uint8")
class_raster

## 5. Write the prediction as a COG and register it

The tiler fetches the file by URL, so after writing it locally, upload it anywhere the tiler
can reach (an Azure blob with a SAS token, an S3 presigned URL, any public file host) and put
that URL in `COG_URL`.

In [ ]:
import rasterio
from rasterio.transform import from_bounds

with rasterio.open(
    "predictions.tif",
    "w",
    driver="COG",
    width=WIDTH,
    height=HEIGHT,
    count=1,
    dtype="uint8",
    crs="EPSG:4326",
    transform=from_bounds(west, south, east, north, WIDTH, HEIGHT),
) as dst:
    dst.write(class_raster, 1)

print("wrote predictions.tif; upload it and set COG_URL below")

In [ ]:
COG_URL = "https://<your-storage>/predictions.tif?<sas-token>"

layer = campaign.register_pred_layer(
    COG_URL,
    classes=campaign.labels,  # class values match label ids, legend shows label names
)
layer

In [ ]:
# Registration runs asynchronously on the server; re-run until status is "ready".
campaign.pred_layers()

Annotators now see the prediction overlay with its legend in the annotation UI and can add
labels where the model is wrong.

## 6. Next round: grow the training set and retrain

`update_samples` appends only samples that are new since the last fetch. The held-out test
set is passed as `exclude`, so it never leaks into training.

In [ ]:
before = len(train)
train = campaign.update_samples(train, exclude=test)
print(f"training set: {before} -> {len(train)} samples")

model.fit(*featurize(train))
print(f"held-out accuracy: {model.score(*featurize(test)):.2f}")

From here the loop repeats: predict over the extent, upload, register the next layer (names
auto-number as `prediction-2`, `prediction-3`, ...), wait for annotators, update, retrain.